In [1]:
import pandas as pd
import numpy as np
import os

# 1. Generate 1 year of hourly timestamps (approx. 8,760 rows)
dates = pd.date_range(start="2025-05-01", end="2026-05-01", freq="h")
df = pd.DataFrame({"timestamp": dates})

# Extract time features for our physics model
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month

# 2. Simulate Weather Features (with realistic day/night and seasonal noise)
# Temperature peaks in the afternoon, humidity drops in the afternoon
df['temperature_c'] = 25 + 10 * np.sin(np.pi * (df['hour'] - 8) / 12) + np.random.normal(0, 2, len(df))
df['humidity_percent'] = 60 - 20 * np.sin(np.pi * (df['hour'] - 8) / 12) + np.random.normal(0, 5, len(df))
df['wind_speed_kmh'] = np.random.uniform(0, 15, len(df))
df['cloud_cover_percent'] = np.random.uniform(0, 100, len(df))

# 3. Calculate the Target Variable: Solar Power Output (Max 500 kW array)
def calculate_solar_output(row):
    # No sun at night
    if row['hour'] < 6 or row['hour'] > 18:
        return 0.0
    
    # Base generation follows a bell curve peaking at noon
    solar_zenith = np.sin(np.pi * (row['hour'] - 6) / 12)
    base_power = 500 * solar_zenith
    
    # Clouds reduce power by up to 80%
    cloud_penalty = 1 - (row['cloud_cover_percent'] / 100 * 0.8)
    
    # High heat actually reduces solar panel efficiency (temp coefficient)
    temp_penalty = 1 - max(0, (row['temperature_c'] - 25) * 0.004)
    
    # Calculate final power and add minor mechanical noise
    actual_power = base_power * cloud_penalty * temp_penalty
    noise = np.random.normal(0, 5)
    
    return max(0, actual_power + noise) # Power can't be negative

# Apply the physics engine to generate our target variable
df['power_output_kw'] = df.apply(calculate_solar_output, axis=1)

# Clean up temporary columns we don't need for the final dataset
df = df.drop(columns=['hour', 'month'])

# 4. Save to our processed data folder
output_path = os.path.join('..', 'data', 'processed', 'historical_solar_data.csv')
df.to_csv(output_path, index=False)

print(f"Successfully generated {len(df)} rows of historical training data!")
display(df.head(15)) # Show the first 15 rows to verify day/night cycles

Successfully generated 8761 rows of historical training data!


,timestamp,temperature_c,humidity_percent,wind_speed_kmh,cloud_cover_percent,power_output_kw
0,2025-05-01 00:00:00,17.296074,80.047372,14.865212,83.048494,0.000000
1,2025-05-01 01:00:00,13.255230,78.160047,1.729049,68.137666,0.000000
2,2025-05-01 02:00:00,14.430102,88.367981,5.094846,60.541197,0.000000
3,2025-05-01 03:00:00,11.955320,69.005818,9.538769,81.797850,0.000000
4,2025-05-01 04:00:00,17.707018,81.893198,7.070706,49.024666,0.000000
5,2025-05-01 05:00:00,19.436366,73.524584,2.818705,93.376192,0.000000
6,2025-05-01 06:00:00,19.916115,69.008236,12.774747,45.427881,0.000000
7,2025-05-01 07:00:00,22.653554,62.751227,3.425407,37.468265,89.862021
8,2025-05-01 08:00:00,24.827587,56.412911,6.994429,27.039338,201.890236
9,2025-05-01 09:00:00,25.047986,55.597481,12.881726,25.834830,271.393022
